# Chapter 11: Understanding Randomness in Structural Engineering
## Why No Two Steel Beams Are Exactly the Same

**Learning Objectives:**
- Explain what makes a process random vs. predictable
- Demonstrate the Law of Large Numbers using structural material tests
- Distinguish short-run variability from long-run stability
- Connect randomness in material properties to how engineers design with safety factors


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(11)

# ASTM A572 Grade 50 structural steel — nominal yield strength 50 ksi
# Real mill data: actual yield strength is normally distributed
# Mean ~55 ksi (mills produce above nominal), SD ~3.5 ksi  (typical coefficient of variation ~6%)
TRUE_MEAN_KSI = 55.0
TRUE_SD_KSI   = 3.5
POPULATION    = np.random.normal(TRUE_MEAN_KSI, TRUE_SD_KSI, 10000)
print('ASTM A572 Grade 50 steel population (10,000 simulated mill coupons)')
print(f'  Nominal minimum yield strength: 50 ksi')
print(f'  Simulated true mean:  {TRUE_MEAN_KSI} ksi')
print(f'  Simulated true SD:    {TRUE_SD_KSI} ksi')
print(f'  Samples below 50 ksi: {(POPULATION < 50).mean()*100:.1f}%  (these would fail the spec)')


## 11.1  Randomness in Structural Materials

When a steel mill produces a W18×97 wide-flange beam, every beam comes out slightly different. The rolling process, cooling rate, and chemical composition of each heat of steel introduce unavoidable variation. This is not sloppy manufacturing — it is physical reality.

**ASTM A572 Grade 50 steel** (one of the most common structural steels, discussed throughout Hibbeler) has a *nominal* minimum yield strength of **50 ksi**. But real mill data shows that:

- The **actual mean** yield strength is closer to **55 ksi** — mills intentionally produce above the minimum
- Individual coupon tests scatter around that mean with a standard deviation of about **3–4 ksi**
- A small percentage of coupons will test *below* 50 ksi even from a compliant heat

This is why structural codes require testing **multiple specimens**, not just one. A single test result is random. The average of many tests is not.

That principle — that averages stabilize even when individual results do not — is called the **Law of Large Numbers**.


## 🔬 Interactive Experiment 1: The Law of Large Numbers in a Steel Mill

Imagine you are a quality-control engineer. You pull steel coupons from a production run and test their yield strength. Each test gives you a random result.

- With **1 test**, your estimate of the batch's average strength could be wildly off.
- With **50 tests**, your estimate is much more reliable.
- With **200 tests**, you have a very accurate picture.

Run the simulation below and watch how the running average of coupon tests converges toward the true mean as the number of tests increases.


In [ ]:
def _lln_plot(n_tests):
    sample = np.random.choice(POPULATION, size=n_tests, replace=False)
    running_avg = np.cumsum(sample) / np.arange(1, n_tests + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: individual test results
    ax1.scatter(range(1, n_tests+1), sample, color='steelblue', alpha=0.5, s=15, label='Individual test')
    ax1.axhline(TRUE_MEAN_KSI, color='black', lw=2, ls='--', label=f'True mean: {TRUE_MEAN_KSI} ksi')
    ax1.axhline(50, color='firebrick', lw=1.5, ls=':', label='ASTM minimum: 50 ksi')
    ax1.set_xlabel('Test number', fontsize=12)
    ax1.set_ylabel('Yield Strength (ksi)', fontsize=12)
    ax1.set_title(f'Individual Coupon Tests  (n = {n_tests})', fontsize=13)
    ax1.legend(fontsize=10)

    # Right: running average
    ax2.plot(range(1, n_tests+1), running_avg, color='steelblue', lw=2, label='Running average')
    ax2.axhline(TRUE_MEAN_KSI, color='black', lw=2, ls='--', label=f'True mean: {TRUE_MEAN_KSI} ksi')
    ax2.fill_between(range(1, n_tests+1),
        TRUE_MEAN_KSI - TRUE_SD_KSI, TRUE_MEAN_KSI + TRUE_SD_KSI,
        alpha=0.12, color='steelblue', label='±1 SD band')
    ax2.set_xlabel('Number of Tests', fontsize=12)
    ax2.set_ylabel('Running Average Yield Strength (ksi)', fontsize=12)
    ax2.set_title('Running Average Converges to True Mean', fontsize=13)
    ax2.legend(fontsize=10)
    final_err = abs(running_avg[-1] - TRUE_MEAN_KSI)
    ax2.annotate(f'Error after {n_tests} tests: {final_err:.2f} ksi',
        xy=(0.03, 0.08), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_n = widgets.IntSlider(value=20, min=1, max=300, step=1,
    description='Number of tests:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out1 = widgets.interactive_output(_lln_plot, {'n_tests': w_n})
display(widgets.VBox([w_n, out1]))


---
## ⚠️  Why Testing Standards Require Multiple Specimens

Before ASTM formalized multi-specimen testing requirements, some engineers accepted single-coupon test results as proof of material compliance. This was dangerous precisely because of the Law of Large Numbers — or rather, its inverse:

> *A single test result tells you almost nothing about the true mean of the population.*

**ASTM A6** (the standard governing structural steel) now requires a minimum number of tension tests per heat of steel, and specifies that the reported yield strength must satisfy statistical requirements — not just a single pass.

The **Quebec Bridge collapse of 1907** is a sobering early example. The bridge's chief engineer, Theodore Cooper, approved the use of heavier steel members based on limited material tests and calculations he performed remotely — without physically being at the site. When a bottom chord member buckled and the bridge collapsed (killing 75 workers), the investigation found that the actual dead load of the structure had been systematically underestimated from the start. A random sample of a few members had been tested and deemed acceptable; the full population of members had not been.

> *Randomness does not go away when you ignore it. It simply waits.*


## 🔬 Interactive Experiment 2: Short-Run Surprises, Long-Run Stability

Individual structural events — a single truck crossing a bridge, a single wind gust — are unpredictable. But the *distribution* of many such events is stable and predictable. This is what allows engineers to design structures for loads they have never seen.

The simulation below shows how the distribution of **maximum observed load** over a bridge stabilizes as more trucks cross. With only a few crossings, the maximum you've seen so far jumps around unpredictably. With thousands of crossings, the pattern of extremes becomes reliable.


In [ ]:
# Truck weights on a highway bridge — roughly log-normal
# AASHTO design truck: 72 kips gross; legal max ~80 kips; overweight trucks exist
TRUCK_MEAN = 50   # kips gross weight (many are lightly loaded)
TRUCK_SD   = 18
TRUCK_POP  = np.abs(np.random.normal(TRUCK_MEAN, TRUCK_SD, 50000))

def _truck_plot(n_crossings):
    sample = np.random.choice(TRUCK_POP, size=n_crossings, replace=True)
    running_max = np.maximum.accumulate(sample)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(range(1, n_crossings+1), running_max, color='darkorange', lw=1.5)
    ax1.axhline(np.percentile(TRUCK_POP, 99), color='firebrick', lw=2, ls='--',
        label='99th percentile truck weight')
    ax1.set_xlabel('Number of Truck Crossings', fontsize=12)
    ax1.set_ylabel('Maximum Observed Weight (kips)', fontsize=12)
    ax1.set_title('Running Maximum Truck Weight', fontsize=13)
    ax1.legend(fontsize=10)

    ax2.hist(sample, bins=40, color='darkorange', edgecolor='white', alpha=0.75)
    ax2.axvline(np.mean(sample), color='black', lw=2, ls='--',
        label=f'Sample mean: {np.mean(sample):.1f} kips')
    ax2.axvline(80, color='firebrick', lw=2, ls=':',
        label='Legal weight limit: 80 kips')
    ax2.set_xlabel('Truck Weight (kips)', fontsize=12)
    ax2.set_ylabel('Number of Trucks', fontsize=12)
    ax2.set_title(f'Distribution of {n_crossings} Truck Weights', fontsize=13)
    ax2.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

w_n2 = widgets.IntSlider(value=50, min=10, max=2000, step=10,
    description='Truck crossings:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out2 = widgets.interactive_output(_truck_plot, {'n_crossings': w_n2})
display(widgets.VBox([w_n2, out2]))


---
## 📋  Chapter 11 Review

| Term | Meaning |
|------|--------|
| **Random process** | Individual outcomes are unpredictable; the long-run pattern is stable |
| **Law of Large Numbers** | As sample size grows, the sample mean converges to the true population mean |
| **Short-run variability** | Individual results fluctuate widely around the true mean |
| **Long-run stability** | The average of many results settles near the true mean |
| **Simulation** | Using a computer to repeat a random process many times and observe the pattern |

**The Big Idea:** Structural engineers rely on the Law of Large Numbers every time they use a code-specified load or material strength. Those values were derived from thousands of tests and measurements. No single test result tells you much — but the average of many tests is a reliable guide for design. Randomness, understood correctly, is not a source of danger. Ignoring it is.
